In [9]:
NOTES_PROMPT = """你是一位专业的会议记录员。将会议记录转换为结构化的 JSON 格式笔记：
{
    "meeting_title": "推断的会议标题",
    "date": "日期（今天或提及的日期）",
    "participants": ["参会人姓名1", "参会人姓名2"],
    "duration_estimate": "时长（X 分钟）",
    "summary": "2-3 句的执行摘要",
    "key_decisions": ["决定 1", "决定 2"],
    "action_items": [
        {"task": "任务描述", "owner": "负责人姓名或待定", "due": "截止日期或时间范围或待定"}
    ],
    "discussion_topics": ["讨论主题 1", "讨论主题 2"],
    "blockers": ["阻塞项 1，无则填 no"],
    "next_meeting": "下次会议时间或待定",
    "follow_up_questions": ["需要跟进解答的问题"]
}
仅返回合法的 JSON，不要包含其他内容。"""

In [10]:
SAMPLE_TRANSCRIPT = """
Sarah：好的大家，我们开始吧。今天是 3 号星期一，参加的有 John、Mike 和 Lisa。

John：谢谢 Sarah。我想主要讨论的是 Q4 产品路线图。我们需要确定功能冻结日期。

Sarah：我觉得我们应该 11 月 15 日之前冻结。这样 QA 团队在假期发布前有 3 周时间测试。

Mike：我可以接受。但我们还需要确定支付集成的方案。Lisa，你那边进展如何？

Lisa：我完成了大概 60%。我需要支付提供商的 API 文档。我已经给他们发了两封邮件，但还没收到回复。

John：我今天会去推动这件事。我会联系我们在 PaymentCo 的客户经理。这目前是个阻塞项。

Sarah：好的，那么 John 负责在今天之内处理 PaymentCo 的升级跟进。Lisa 继续推进支付集成，目标 11 月 10 日完成。

Mike：Lisa 拿到草案后我可以帮忙测试。暂定我 11 月 11 日开始测试。

Sarah：好的。另外我们决定把这个发布版本中的社交登录功能砍掉。现在加入风险太大了。

John：同意。我们会把它放到 Q1 的 backlog（待办列表）里。

Sarah：还有其他阻塞项吗？没有了？好的，下周同一时间，11 月 10 日再开。
"""

In [11]:
import re
import json

def parse_json_response(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if match:
        cleaned = match.group(0)
    return json.loads(cleaned)

In [15]:
from dotenv import load_dotenv
import os

load_dotenv()

model = os.getenv('MODEL_NAME', 'gpt-5.5')

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

def generate_meeting_notes(transcript: str) -> dict:
    llm = ChatOpenAI(model=model, temperature=0, extra_body={'enable_thinking': False})
    messages = [
        SystemMessage(content=NOTES_PROMPT),
        HumanMessage(content=f"会议转录文本:\n\n{transcript}"),
    ]
    response = llm.invoke(messages)
    return parse_json_response(response.content)

In [ ]:
from datetime import date

def format_notes(notes: dict) -> str:
    lines = [
        f"# {notes.get('meeting_title', '会议主题')}",
        f"**会议时间:** {notes.get('date', date.today().isoformat())}  |  **会议时长:** {notes.get('duration_estimate', 'N/A')}",
        f"**参会成员:** {', '.join(notes.get('participants', []))}",
        "",
        "## 会议纪要",
        notes.get("summary", ""),
        "",
        "## 关键决策",
        *[f"- {d}" for d in notes.get("key_decisions", [])],
        "",
        "## 待办事项",
    ]
    for item in notes.get("action_items", []):
        lines.append(f"- [ ] **{item.get('task', 'N/A')}** — 负责人: {item.get('owner', '待定')} | 截止日期: {item.get('due', '待定')}")

    if notes.get("blockers"):
        lines += ["", "## 阻塞项", *[f"- {b}" for b in notes["blockers"]]]

    if notes.get("next_meeting") and notes["next_meeting"] != "TBD":
        lines += ["", f"**下次会议:** {notes['next_meeting']}"]

    return "\n".join(lines)

In [20]:
import json

notes = generate_meeting_notes(transcript=SAMPLE_TRANSCRIPT)

print(json.dumps(obj=notes, ensure_ascii=False, indent=2))

formatted = format_notes(notes=notes)

print("=" * 60)
print(formatted)
print("=" * 60)

{
  "meeting_title": "Q4 产品路线图与支付集成讨论",
  "date": "11月3日",
  "participants": [
    "Sarah",
    "John",
    "Mike",
    "Lisa"
  ],
  "duration_estimate": "15分钟",
  "summary": "会议确定了Q4产品功能冻结日期为11月15日，并决定将社交登录功能移至Q1待办列表以规避风险。重点讨论了支付集成的进展，确认了相关任务的负责人及截止日期。",
  "key_decisions": [
    "功能冻结日期定为11月15日",
    "从当前发布版本中移除社交登录功能，放入Q1 backlog"
  ],
  "action_items": [
    {
      "task": "联系PaymentCo客户经理以获取API文档",
      "owner": "John",
      "due": "今天之内"
    },
    {
      "task": "继续推进支付集成工作",
      "owner": "Lisa",
      "due": "11月10日"
    },
    {
      "task": "开始测试支付集成草案",
      "owner": "Mike",
      "due": "11月11日"
    }
  ],
  "discussion_topics": [
    "Q4产品路线图与功能冻结",
    "支付集成方案及API文档获取",
    "社交登录功能的去留决策"
  ],
  "blockers": [
    "缺少支付提供商(PaymentCo)的API文档"
  ],
  "next_meeting": "11月10日同一时间",
  "follow_up_questions": [
    "PaymentCo何时能回复API文档？"
  ]
}
# Q4 产品路线图与支付集成讨论
**会议时间:** 11月3日  |  **会议时长:** 15分钟
**参会成员:** Sarah, John, Mike, Lisa

## 会议纪要
会议确定了Q4产品功能冻结日期为11月15日，并决定将社交登录功能移至Q